Importação de bibliotecas para abrir arquivos e medir o tempo de execução, além do multithreading

In [1]:
import time
import random
import threading
import logging

In [2]:
# # Gerar números aleatórios e salvar em um arquivo
# def gerar_numeros_aleatorios(arquivo: str, quantidade:int) -> None:
#     """
#     Gera uma quantidade especificada de números aleatórios e salva em um arquivo.
#     """
#     numeros = [str(random.randint(1, 10000)) for _ in range(quantidade)]
#     with open(arquivo, 'w') as f:
#         f.write(','.join(numeros))  # Usar espaço como separador
#     print(f"Números aleatórios gerados e salvos em {arquivo}")

### Implementação Insertion Sort
* O Insertion Sort é um algoritmo de ordenação que constrói a lista ordenada um elemento de cada vez.
* Ele é mais eficiente para listas pequenas ou listas que já estão quase ordenadas.
* O algoritmo funciona dividindo a lista em duas partes: a parte ordenada e a parte não ordenada.
* Ele pega um elemento da parte não ordenada e o insere na posição correta na parte ordenada.
* Varia entre O(n^2) e O(n) dependendo da ordenação inicial da lista.

In [3]:
def insertion_sort(lista: list) -> tuple[list, float]:
    """
    Ordena uma lista usando o algoritmo Insertion Sort.

    Args:
        lista: A lista a ser ordenada.

    Returns:
        Tupla com a lista ordenada e o tempo de execução.
    """
    if len(lista) == 0:
        return lista, 0.0
    
    inicio = time.time()
    
    for i in range(1, len(lista)):
        chave = lista[i]
        j = i - 1
        while j >= 0 and lista[j] > chave:
            lista[j + 1] = lista[j]
            j -= 1
        lista[j + 1] = chave
    
    fim = time.time()
    tempo_ordenacao = fim - inicio
    logging.info(f"Tempo total de execução do Insertion Sort: {tempo_ordenacao:.6f} segundos.")
    
    return lista, tempo_ordenacao

In [4]:
def insertion_sort_multithreaded(lista: list, num_threads: int, logger) -> tuple[list, float]:
    """
    Ordena uma lista usando o algoritmo Insertion Sort com múltiplas threads.

    Args:
        lista: A lista a ser ordenada.
        num_threads: O número de threads a serem usadas.
        logger: Logger para registrar informações.

    Returns:
        Tupla com a lista ordenada e o tempo de execução.
    """
    if num_threads <= 0:
        logger.error("O número de threads deve ser maior que zero.")
        raise ValueError("O número de threads deve ser maior que zero.")
    
    logger.info(f"Iniciando Insertion Sort com {num_threads} threads.")
    inicio = time.time()
    
    tamanho = len(lista)
    threads = []
    partes = [lista[i * tamanho // num_threads:(i + 1) * tamanho // num_threads] for i in range(num_threads)]

    # Função auxiliar para ordenar sem retornar tempo
    def insertion_sort_simples(sublista):
        for i in range(1, len(sublista)):
            chave = sublista[i]
            j = i - 1
            while j >= 0 and sublista[j] > chave:
                sublista[j + 1] = sublista[j]
                j -= 1
            sublista[j + 1] = chave

    for i in range(num_threads):
        thread = threading.Thread(target=insertion_sort_simples, args=(partes[i],))
        threads.append(thread)
        thread.start()
        logger.info(f"Thread {i + 1} iniciada para ordenar a parte {i + 1} com {len(partes[i])} elementos.")
    
    logger.info("Threads iniciadas para ordenação.")
    
    for thread in threads:
        thread.join()
    logger.info("Todas as threads concluídas.")

    # Combinar as partes ordenadas
    resultado = []
    for parte in partes:
        resultado.extend(parte)
    
    fim = time.time()
    tempo_ordenacao = fim - inicio
    logger.info(f"Tempo total de execução: {tempo_ordenacao:.6f} segundos.")
    
    return resultado, tempo_ordenacao

### Bucketsort
* Não é um método de ordenação, e sim um método de distribuição.
* O Bucket Sort é um algoritmo de ordenação que distribui os elementos em vários "baldes" (buckets).
* Cada balde é ordenado individualmente, geralmente usando um algoritmo de ordenação simples, como o Insertion Sort.
* O Bucket Sort é eficiente para listas com valores uniformemente distribuídos.
* Por exemplo, se tivermos uma lista entre 0 e 25, podemos criar 5 baldes: [0-5], [6-10], [11-15], [16-20], [20-25].
    * Para dividir os elementos, podemos usar a fórmula: bucket = elemento / 5
    * Arredondamos para cima, pois o bucket 0 é o [0-5], o bucket 1 é o [6-10], e assim por diante.
* Após a distribuição, ordenamos cada balde individualmente.
* Após a ordenação, concatenamos os baldes para obter a lista ordenada.

In [5]:
# Método Bucket Sort para auxiliar na ordenação
def separar_buckets(lista: list,qnt_buckets:int) -> list:
    """
    Separa os elementos da lista em buckets.

    Args:
        lista: A lista a ser separada.
        qnt_buckets: A quantidade de buckets a serem criados.

    Returns:
        A quantidade de buckets e a lista de buckets.
    """
    if len(lista) == 0:
        return []
    max_valor = max(lista)
    min_valor = min(lista)
    intervalo = (max_valor - min_valor) / qnt_buckets
    buckets = [[] for _ in range(qnt_buckets)]
    for numero in lista:
        # Determinar o índice do bucket
        if numero == max_valor:  # Garantir que o maior valor vá para o último bucket
            indice_bucket = qnt_buckets - 1
        else:
            indice_bucket = int((numero - min_valor) / intervalo)
        buckets[indice_bucket].append(numero)

    return buckets
    
def bucket_sort(lista: list, num_buckets: int) -> tuple[list, float]:
    """
    Ordena uma lista usando o algoritmo Bucket Sort.

    Args:
        lista: A lista a ser separada.
        num_buckets: A quantidade de buckets a serem criados.

    Returns:
        Tupla com a lista ordenada e o tempo de execução.
    """
    if len(lista) == 0:
        return lista, 0.0

    inicio = time.time()
    
    # Separar os elementos em buckets
    buckets = separar_buckets(lista, num_buckets)

    # Ordenar cada bucket individualmente (sem retorno de tempo)
    for i in range(len(buckets)):
        buckets[i], _ = insertion_sort(buckets[i])

    # Concatenar os buckets ordenados
    resultado = []
    for bucket in buckets:
        resultado.extend(bucket)
    
    fim = time.time()
    tempo_ordenacao = fim - inicio
    logging.info(f"Tempo total de execução do Bucket Sort: {tempo_ordenacao:.6f} segundos.")

    return resultado, tempo_ordenacao
  

In [6]:
def bucket_sort_multithreaded(lista: list, num_threads: int, logger, num_buckets: int) -> tuple[list, float]:
    """
    Ordena uma lista usando o algoritmo Bucket Sort com múltiplas threads.

    Args:
        lista: A lista a ser ordenada.
        num_threads: O número de threads a serem usadas.
        logger: Logger para registrar informações.
        num_buckets: Número de buckets a serem criados.

    Returns:
        Tupla com a lista ordenada e o tempo de execução.
    """
    if len(lista) == 0:
        return lista, 0.0

    if num_threads <= 0:
        logger.error("O número de threads deve ser maior que zero.")
        raise ValueError("O número de threads deve ser maior que zero.")
        
    logger.info(f"Iniciando Bucket Sort com {num_threads} threads.")
    inicio = time.time()
    
    # Separar os elementos em buckets
    buckets = separar_buckets(lista, num_buckets)
    logger.info(f"Separados {len(buckets)} buckets para ordenação.")
    
    # Função auxiliar para ordenar sem retornar tempo
    def insertion_sort_simples(sublista):
        for i in range(1, len(sublista)):
            chave = sublista[i]
            j = i - 1
            while j >= 0 and sublista[j] > chave:
                sublista[j + 1] = sublista[j]
                j -= 1
            sublista[j + 1] = chave

    # Dividir os buckets entre as threads
    def ordenar_grupo_buckets(buckets_grupo):
        for bucket in buckets_grupo:
            insertion_sort_simples(bucket)
    
    # Dividir os buckets em grupos para as threads
    buckets_por_thread = [[] for _ in range(num_threads)]
    for i, bucket in enumerate(buckets):
        buckets_por_thread[i % num_threads].append(bucket)
    
    threads = []
    for i in range(num_threads):
        thread = threading.Thread(target=ordenar_grupo_buckets, args=(buckets_por_thread[i],))
        threads.append(thread)
        thread.start()
        logger.info(f"Thread {i + 1} iniciada para ordenar {len(buckets_por_thread[i])} buckets.")
    
    for thread in threads:
        thread.join()
    logger.info("Todas as threads concluídas.")

    # Concatenar os buckets ordenados
    resultado = []
    for bucket in buckets:
        resultado.extend(bucket)
    
    fim = time.time()
    tempo_ordenacao = fim - inicio
    logger.info(f"Tempo total de execução: {tempo_ordenacao:.6f} segundos.")

    return resultado, tempo_ordenacao

In [7]:
# Abrir o arquivo "Numeros.txt" e ler os números
def abrir_arquivo(arquivo: str) -> list:
    """
    Lê os números de um arquivo e retorna uma lista.

    Args:
        arquivo: O caminho do arquivo a ser lido.

    Returns:
        Uma lista com os números lidos do arquivo.
    """
    with open(arquivo, 'r') as f:
        # Os numeros estão separados por vírgula
        numeros = f.read().strip().split('\n')
    # Converter os números de string para inteiro
    numeros = [int(num) for num in numeros if num.isdigit()]
    return numeros

In [8]:
# Função para criar o arquivo de saída com os números ordenados, o tempo de execução e o método (Insertion Sort com ou sem bucketsort) e técnica de threading
def criar_arquivo_saida(arquivo_output:str, worA:str, num_elementos:int, tempo: float, metodo: str, multiThreading:bool, num_buckets:int,execucao:int)-> None:
    """
    Cria um arquivo de saída com os números ordenados, o tempo de execução e o método utilizado.

    Args:
        numeros: A lista de números ordenados.
        tempo: O tempo de execução do algoritmo.
        metodo: O método utilizado para ordenar os números.
    """
    with open(arquivo_output, worA) as f:
        # Verificar se o arquivo já existe
        if worA == 'w':
            f.write('Quantidade de Elementos,Tempo de Execucao (s),Metodo,Multithreading,Numero Buckets,Execução\n',)
        # Escrever os dados no arquivo
        f.write(f'{num_elementos},{tempo},{metodo},{"Sim" if multiThreading else "Nao"},{num_buckets},{execucao}\n')

In [9]:
def executar_ordenacao(algoritmo: str, usar_threads: bool = False, num_threads: int = 4, 
                       arquivo_entrada: str = '../Arquivos/input/numerosSimples.txt', 
                       arquivo_saida: str = '../Arquivos/output/outputSimples.csv',
                       modo_escrita: str = 'a',
                       num_buckets: int = 0,
                       execucao: int = 1) -> list:
    """
    Função generalizada para executar algoritmos de ordenação com ou sem multithreading.
    """
    # Configurar logger
    logger = logging.getLogger(f"{algoritmo}_{usar_threads}")
    logger.setLevel(logging.INFO)
    if not logger.handlers:
        handler = logging.StreamHandler()
        formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
        handler.setFormatter(formatter)
        logger.addHandler(handler)
    
    # Abrir o arquivo e ler os números
    numeros = abrir_arquivo(arquivo_entrada)
    print(f"Números lidos do arquivo: {len(numeros)} números")
    
    # Escolher o algoritmo e se deve usar multithreading
    if algoritmo.lower() == 'insertion':
        if usar_threads:
            logger.info(f"Usando {num_threads} threads para Insertion Sort.")
            numeros_ordenados, tempo_execucao = insertion_sort_multithreaded(numeros.copy(), num_threads, logger)
            metodo = 'Insertion Sort'
        else:
            logger.info("Usando Insertion Sort sem multithreading.")
            numeros_ordenados, tempo_execucao = insertion_sort(numeros.copy())
            metodo = 'Insertion Sort'
    elif algoritmo.lower() == 'bucket':
        if usar_threads:
            logger.info(f"Usando {num_threads} threads para Bucket Sort.")
            numeros_ordenados, tempo_execucao = bucket_sort_multithreaded(numeros.copy(), num_threads, logger, num_buckets)
            metodo = 'Bucket Sort'
        else:
            logger.info("Usando Bucket Sort sem multithreading.")
            numeros_ordenados, tempo_execucao = bucket_sort(numeros.copy(), num_buckets)
            metodo = 'Bucket Sort'
    else:
        logger.error(f"Algoritmo {algoritmo} não reconhecido.")
        raise ValueError(f"Algoritmo {algoritmo} não reconhecido. Use 'insertion' ou 'bucket'.")
    
    print(f"Tempo de execução: {tempo_execucao:.6f} segundos")
    
    # Criar o arquivo de saída com os resultados
    criar_arquivo_saida(arquivo_saida, modo_escrita, len(numeros), tempo_execucao, metodo, usar_threads, num_buckets,execucao)
    
    return numeros_ordenados

In [ ]:
# Criar um lock global
ordenacao_lock = threading.Lock()
def executar_todos_metodos(execucao:int):
    """
    Executa todos os métodos de ordenação e salva os resultados.
    Usa um lock para garantir que apenas um método seja executado por vez.
    
    Args:
        arquivo_entrada: O caminho do arquivo de entrada.
        arquivo_saida: O caminho do arquivo de saída.
    """
    dirs_entrada = {
        'Aleatorios': '../Arquivos/input/Aleatorios',
        'Ordenados': '../Arquivos/input/Ordenados',
        'Decrescentes': '../Arquivos/input/Decrescentes',
        'Parcialmente Ordenados': '../Arquivos/input/ParcialmenteOrdenados'
    }
    
    arquivos_entradas = {}
    
    num_elementos = [750000,1000000,1250000,1500000,2000000]
    for tipo, caminho in dirs_entrada.items():
        for i in num_elementos:
            inicio = tipo.lower()[0] if ' ' not in tipo else tipo.lower().split(" ")[0][0] + tipo.lower().split(" ")[1][0]        
            arquivo = f'{caminho}/{inicio+str(i)}.txt'
            arquivos_entradas[f'{tipo} {i}'] = arquivo
    
    
    # Lista de arquivos de entrada que o insertion sort não executa
    insertion_exclude = ['750000', '1000000', '1250000', '1500000', '2000000']
    # Porém, caso seja ordenado ou parcialmente ordenado, o insertion sort executa até 1M

    # Adquirir o lock global para garantir execução exclusiva
    with ordenacao_lock:
        print("Iniciando execução de todos os métodos de ordenação...")
        
        for tipo, arquivo_entrada in arquivos_entradas.items():
            print(tipo, arquivo_entrada)
            print(f"\n=== Executando métodos para {tipo[0]} ===")
            # Criar o arquivo de saída específico para cada tipo
            arquivo_saida = f'../Arquivos/output/output_{tipo.split(" ")[0]}.csv'
            
            # Verificar se o arquivo de entrada é um dos que o Insertion Sort não executa
            if tipo.split(" ")[1] not in insertion_exclude:
                # Insertion Sort sem threads
                print("\n=== Executando Insertion Sort sem threads ===")
                executar_ordenacao('insertion', False, arquivo_entrada=arquivo_entrada, 
                                arquivo_saida=arquivo_saida,execucao=execucao)
            
                # Insertion Sort com threads
                print("\n=== Executando Insertion Sort com threads ===")
                executar_ordenacao('insertion', True, arquivo_entrada=arquivo_entrada, 
                                arquivo_saida=arquivo_saida,execucao=execucao)
            
            # Bucket Sort sem threads, 10 buckets
            print("\n=== Executando Bucket Sort sem threads ===")
            executar_ordenacao('bucket', False, arquivo_entrada=arquivo_entrada, 
                              arquivo_saida=arquivo_saida,num_buckets=10,execucao=execucao)
        
            # Bucket Sort sem threads, 100 buckets
            print("\n=== Executando Bucket Sort sem threads ===")
            executar_ordenacao('bucket', False, arquivo_entrada=arquivo_entrada, 
                              arquivo_saida=arquivo_saida,num_buckets=100,execucao=execucao)
            
            # Bucket Sort sem threads, 1000 buckets
            print("\n=== Executando Bucket Sort sem threads ===")
            executar_ordenacao('bucket', False, arquivo_entrada=arquivo_entrada, 
                              arquivo_saida=arquivo_saida,num_buckets=1000,execucao=execucao)
            
            # Bucket Sort com threads, 10 buckets
            print("\n=== Executando Bucket Sort com threads ===")
            executar_ordenacao('bucket', True, arquivo_entrada=arquivo_entrada, 
                              arquivo_saida=arquivo_saida, num_buckets=10,execucao=execucao)
            # Bucket Sort com threads, 100 buckets
            print("\n=== Executando Bucket Sort com threads ===")
            executar_ordenacao('bucket', True, arquivo_entrada=arquivo_entrada, 
                              arquivo_saida=arquivo_saida, num_buckets=100,execucao=execucao)
            # Bucket Sort com threads, 1000 buckets
            print("\n=== Executando Bucket Sort com threads ===")
            executar_ordenacao('bucket', True, arquivo_entrada=arquivo_entrada, 
                              arquivo_saida=arquivo_saida, num_buckets=1000,execucao=execucao)
        
        
        print("\nTodos os métodos de ordenação foram executados com sucesso.")

In [11]:
for i in range(1, 6):
    print(f"Executando teste {i}...")
    executar_todos_metodos(i)

Executando teste 1...
Iniciando execução de todos os métodos de ordenação...
Aleatorios 750000 ../Arquivos/input/Aleatorios/a750000.txt

=== Executando métodos para A ===

=== Executando Bucket Sort sem threads ===


2025-07-16 08:44:11,993 - bucket_False - INFO - Usando Bucket Sort sem multithreading.


Números lidos do arquivo: 750000 números


KeyboardInterrupt: 